In [1]:
!pip install transformers torch gradio

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
# Load model and tokenizer
model_name = "google/gemma-2-2b-it"
token = "HUGGINGFACE_API_KEY"

# Check if CUDA is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model and tokenizer with token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

Using device: cuda


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [8]:
# Function to generate chatbot response without chat history
def chatbot_response(user_input):
    # Tokenize input and move to GPU
    inputs = tokenizer(user_input, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate response
    outputs = model.generate(**inputs)

    # Decode output
    bot_reply = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Ensure the bot doesn't repeat the user input
    if bot_reply.startswith(user_input):
        bot_reply = bot_reply[len(user_input):].strip()

    return bot_reply

In [9]:
chatbot_response("Hi how are you,my name is besho what is yours?")

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1612: UserWarning: Using the model-agnostic default `max_length` (=35) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


"Hello Besho, it's nice to meet you! I'm doing well, thank"

In [10]:
def chatbot_interface(user_input, history):
    response = chatbot_response(user_input)
    history.append((user_input, response))
    return history, ""

# Creating the Gradio Interface
demo = gr.Blocks()

with demo:
    gr.Markdown("## GenAI Chatbot")
    chatbot = gr.Chatbot()
    with gr.Row():
        user_input = gr.Textbox(placeholder="Type your message here...", lines=1)
        send_button = gr.Button("Send")

    history = gr.State([])

    send_button.click(chatbot_interface, inputs=[user_input, history], outputs=[chatbot, user_input])
    user_input.submit(chatbot_interface, inputs=[user_input, history], outputs=[chatbot, user_input])

# Launch the chatbot
demo.launch(share = True)

/tmp/ipykernel_3988/2678549946.py:11: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_3988/2678549946.py:11: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://256c2e02d7aaaee629.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
